# CAMEL-AI SOC Generator - 500K Pairs

Usa CAMEL-AI framework con dos agentes (User + Assistant) para generar 500K conversaciones
sintéticas en español. Formato U:/B: + ShareGPT JSON.

**Backend**: Ollama + Phi-3-mini (local en GPU Kaggle P100)
**Output**: 30 archivos `chat_06.txt` a `chat_35.txt`

In [ ]:
# ============================================================
# INSTALACIÓN
# ============================================================
!pip install -q camel-ai aiohttp tqdm

# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama serve > /tmp/ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull phi3:mini

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ============================================================
# CAMEL-AI CONFIG - Dos Agentes Role-Playing
# ============================================================
import json, os, random, hashlib, asyncio, aiohttp
from pathlib import Path
from tqdm.notebook import tqdm

class CAMELConfig:
    TARGET_PAIRS = 500000
    NUM_FILES = 30
    OUTPUT_DIR = "/kaggle/working/soc_corpus"
    
    # Ollama config
    OLLAMA_BASE = "http://localhost:11434"
    MODEL = "phi3:mini"
    
    # CAMEL-AI Role definitions (Inception Prompting)
    USER_ROLES = [
        ("estudiante", "Eres un estudiante universitario de 20 años. Hablas de forma informal, usas jerga, a veces impreciso. Preguntas sobre programación, ciencia, vida cotidiana."),
        ("profesional", "Eres un ingeniero de software senior de 35 años. Hablas técnico, preciso, usas terminología. Preguntas sobre arquitectura, devops, performance."),
        ("curioso", "Eres una persona curiosa de 28 años. Te fascina todo. Preguntas sobre ciencia, historia, filosofía, tecnología."),
        ("frustrado", "Eres un usuario frustrado con problemas técnicos. Hablas directo, sin contexto. Reportas errores, bugs, crashes."),
        ("creativo", "Eres un escritor creativo de 30 años. Pides historias, poemas, ideas creativas. Imaginativo, detallista."),
        ("docente", "Eres un profesor de 45 años. Explicas paso a paso, paciente, pedagógico. Pides explicaciones, ejemplos, ejercicios."),
        ("casual", "Eres una persona casual de 25 años. Hablas coloquial, abreviado. Preguntas sobre vida, tecnología, entretenimiento."),
        ("experto", "Eres un investigador de 40 años. Preguntas profundas, técnicas, referencias a papers. State-of-the-art, benchmarks."),
    ]
    
    ASSISTANT_ROLES = [
        ("profesor", "Eres un profesor universitario experto. Explicas con claridad, usas ejemplos, admites cuando no sabes. Español natural."),
        ("ingeniero", "Eres un ingeniero de software senior. Respuestas técnicas, precisas, con código y ejemplos prácticos."),
        ("experto", "Eres un experto en múltiples campos. Respuestas profundas, con referencias, análisis crítico."),
        ("amigo", "Eres un amigo cercano. Respuestas informales, empáticas, con consejos prácticos."),
        ("tutor", "Eres un tutor paciente. Explicas paso a paso, con ejercicios, sin asumir conocimiento previo."),
    ]
    
    # Temas para diversidad
    TOPICS = [
        "python", "javascript", "rust", "go", "react", "vue", "angular",
        "docker", "kubernetes", "aws", "azure", "linux", "git",
        "machine_learning", "deep_learning", "ia", "nlp", "cv",
        "matematicas", "fisica", "quimica", "biologia", "astronomia",
        "filosofia", "historia", "psicologia", "economia",
        "salud", "nutricion", "ejercicio", "meditacion",
        "cocina", "viajes", "musica", "arte", "fotografia",
        "emprendimiento", "marketing", "finanzas", "inversiones",
        "relaciones", "comunicacion", "liderazgo", "productividad",
        "educacion", "idiomas", "certificaciones", "carrera",
    ]

CONFIG = CAMELConfig()
print(f"CAMEL-AI Config: {CONFIG.TARGET_PAIRS} pares, {CONFIG.NUM_FILES} archivos")
print(f"User roles: {len(CONFIG.USER_ROLES)}, Assistant roles: {len(CONFIG.ASSISTANT_ROLES)}")

In [ ]:
# ============================================================
# CAMEL-AI DUAL AGENT GENERATOR (Inception Prompting)
# ============================================================

class CAMELSOCGenerator:
    """
    Genera conversaciones usando CAMEL-AI Inception Prompting:
    - User Agent: role-playea un usuario con persona específica
    - Assistant Agent: role-playea un asistente experto
    - Ambos generan conversación multi-turno realista
    """
    
    def __init__(self, config):
        self.config = config
        self.session = None
        self.conversations = []
        self.seen_hashes = set()
    
    async def __aenter__(self):
        self.session = aiohttp.ClientSession(
            timeout=aiohttp.ClientTimeout(total=180)
        )
        return self
    
    async def __aexit__(self, *args):
        if self.session:
            await self.session.close()
    
    def _get_system_prompt(self, user_role, assistant_role, topic):
        """CAMEL-AI Inception Prompting: define roles para ambos agentes"""
        user_name, user_desc = user_role
        asst_name, asst_desc = assistant_role
        
        return f"""<USER_AGENT>
Rol: {user_name}
Descripcion: {user_desc}
Tema: {topic}
Instrucciones:
- Inicia la conversación con una pregunta natural sobre {topic}
- Responde de forma realista, como lo haría un {user_name}
- Usa lenguaje natural, coloquial si es apropiado
- Puedes hacer seguimiento, pedir aclaraciones, expresar dudas
- Mantén 3-8 turnos de conversación
</USER_AGENT>

<ASSISTANT_AGENT>
Rol: {asst_name}
Descripcion: {asst_desc}
Instrucciones:
- Responde de forma útil, clara y natural
- Admite cuando no sabes algo
- Usa ejemplos prácticos cuando sea posible
- Español neutro/latinoamericano
- Respuestas de longitud variable (cortas y largas)
- NO uses formato markdown, solo texto plano
</ASSISTANT_AGENT>"""
    
    async def generate_conversation(self, user_role, asst_role, topic):
        """Genera una conversación completa usando dos agentes CAMEL-AI"""
        
        system = self._get_system_prompt(user_role, asst_role, topic)
        
        # Step 1: User agent genera primera pregunta
        user_prompt = f"""Eres {user_role[0]}. Tema: {topic}.
        Genera UNA pregunta natural y realista sobre {topic}.
        Solo la pregunta, sin explicaciones adicionales."""
        
        first_user_msg = await self._call_ollama(
            f"{system}\n\n{user_prompt}",
            temperature=0.9
        )
        
        if not first_user_msg:
            return None
        
        # Step 2: Alternar User/Assistant para multi-turno
        turns = [{"role": "user", "content": first_user_msg.strip()}]
        
        num_turns = random.randint(3, 8)
        
        for i in range(num_turns - 1):
            # Assistant responde
            asst_prompt = f"""Conversación hasta ahora:
{'\n'.join([f"{'User' if t['role']=='user' else 'Assistant'}: {t['content']}" for t in turns])}

Continúa como {asst_role[0]}. Responde de forma natural y útil."""
            
            asst_response = await self._call_ollama(
                f"{system}\n\n{asst_prompt}",
                temperature=0.7
            )
            
            if not asst_response:
                break
            
            turns.append({"role": "assistant", "content": asst_response.strip()})
            
            # User sigue (si no es el último turno)
            if i < num_turns - 2:
                user_followup = f"""Conversación hasta ahora:
{'\n'.join([f"{'User' if t['role']=='user' else 'Assistant'}: {t['content']}" for t in turns])}

Continúa como {user_role[0]}. Haz una pregunta de seguimiento o reacciona a la respuesta."""
                
                user_next = await self._call_ollama(
                    f"{system}\n\n{user_followup}",
                    temperature=0.9
                )
                
                if user_next:
                    turns.append({"role": "user", "content": user_next.strip()})
        
        return turns
    
    async def _call_ollama(self, prompt, temperature=0.7):
        """Llama a Ollama API"""
        payload = {
            "model": self.config.MODEL,
            "prompt": prompt,
            "temperature": temperature,
            "top_p": 0.9,
            "stream": False,
            "options": {"num_predict": 1024}
        }
        
        try:
            async with self.session.post(
                f"{self.config.OLLAMA_BASE}/api/generate",
                json=payload
            ) as resp:
                data = await resp.json()
                return data.get("response", "")
        except Exception as e:
            return None
    
    def _is_quality(self, turns):
        """Filtra conversaciones de baja calidad"""
        if len(turns) < 4:
            return False
        for t in turns:
            if len(t["content"]) < 10:
                return False
        # Verificar que no sea repetitivo
        user_texts = [t["content"] for t in turns if t["role"] == "user"]
        if len(set(user_texts)) != len(user_texts):
            return False
        return True
    
    def _hash_conv(self, turns):
        text = "".join(t["content"] for t in turns)
        return hashlib.md5(text.encode()).hexdigest()[:16]
    
    async def generate_batch(self, batch_size):
        """Genera un batch de conversaciones"""
        tasks = []
        for _ in range(batch_size):
            user_role = random.choice(self.config.USER_ROLES)
            asst_role = random.choice(self.config.ASSISTANT_ROLES)
            topic = random.choice(self.config.TOPICS)
            tasks.append(self.generate_conversation(user_role, asst_role, topic))
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        valid = []
        for r in results:
            if isinstance(r, list) and self._is_quality(r):
                h = self._hash_conv(r)
                if h not in self.seen_hashes:
                    self.seen_hashes.add(h)
                    valid.append(r)
        return valid
    
    def save_conversations(self, conversations, suffix="final"):
        """Guarda conversaciones en formato U:/B:"""
        Path(self.config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        
        # Save JSONL
        with open(f"{self.config.OUTPUT_DIR}/soc_{suffix}.jsonl", "w", encoding="utf-8") as f:
            for conv in conversations:
                f.write(json.dumps(conv, ensure_ascii=False) + "\n")
        
        # Save ShareGPT
        sharegpt = []
        for conv in conversations:
            msgs = [{"from": t["role"], "value": t["content"]} for t in conv]
            sharegpt.append({"conversations": msgs})
        
        with open(f"{self.config.OUTPUT_DIR}/soc_{suffix}_sharegpt.json", "w", encoding="utf-8") as f:
            json.dump(sharegpt, f, ensure_ascii=False, indent=2)
    
    def split_to_files(self, conversations):
        """Divide en 30 archivos chat_06.txt a chat_35.txt"""
        per_file = len(conversations) // self.config.NUM_FILES
        remainder = len(conversations) % self.config.NUM_FILES
        
        start = 0
        for i in range(self.config.NUM_FILES):
            end = start + per_file + (1 if i < remainder else 0)
            batch = conversations[start:end]
            
            filename = f"chat_{6 + i:02d}.txt"
            filepath = os.path.join(self.config.OUTPUT_DIR, filename)
            
            with open(filepath, "w", encoding="utf-8") as f:
                for conv in batch:
                    for turn in conv:
                        role = "U" if turn["role"] == "user" else "B"
                        f.write(f"{role}: {turn['content']}\n")
                    f.write("\n")
            
            size_kb = os.path.getsize(filepath) // 1024
            print(f"  {filename}: {len(batch):,} convs ({size_kb} KB)")
            start = end

print("CAMEL-AI SOC Generator loaded")

In [ ]:
# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

async def main():
    async with CAMELSOCGenerator(CONFIG) as gen:
        pbar = tqdm(total=CONFIG.TARGET_PAIRS, desc="CAMEL-AI SOC")
        
        while len(gen.conversations) < CONFIG.TARGET_PAIRS:
            batch = await gen.generate_batch(16)
            
            if batch:
                gen.conversations.extend(batch)
                pbar.update(len(batch))
                
                # Checkpoint cada 10K
                if len(gen.conversations) % 10000 < 16:
                    gen.save_conversations(gen.conversations, f"checkpoint_{len(gen.conversations)}")
                    print(f"\nCheckpoint: {len(gen.conversations):,} conversaciones")
        
        # Split en 30 archivos
        print(f"\nDividiendo {len(gen.conversations):,} conversaciones en {CONFIG.NUM_FILES} archivos...")
        gen.split_to_files(gen.conversations)
        
        # Guardado final
        gen.save_conversations(gen.conversations, "final")
        
        print(f"\n{'='*60}")
        print(f"COMPLETADO: {len(gen.conversations):,} conversaciones CAMEL-AI")
        print(f"Output: {CONFIG.OUTPUT_DIR}")
        
        return gen.conversations

conversations = await main()

In [ ]:
# ============================================================
# VERIFICACIÓN
# ============================================================
import os

output_dir = CONFIG.OUTPUT_DIR
files = sorted([f for f in os.listdir(output_dir) if f.startswith("chat_") and f.endswith(".txt")])
total_pairs = 0

for f in files:
    filepath = os.path.join(output_dir, f)
    size_kb = os.path.getsize(filepath) // 1024
    with open(filepath, "r") as fp:
        pairs = fp.read().count("U: ")
    total_pairs += pairs
    print(f"{f}: {pairs:,} pares ({size_kb} KB)")

print(f"\nTotal U:/B: pairs: {total_pairs:,}")

In [ ]:
# ============================================================
# SUBIR A GITHUB
# ============================================================

!git config --global user.email "kaggle@rubidium.ai"
!git config --global user.name "Kaggle Bot"

repo_dir = "/kaggle/working/rubidium-api"
if not os.path.exists(repo_dir):
    !git clone https://github.com/diegovelandiabarajas1-lang/rubidium-api.git {repo_dir}

resources_dir = f"{repo_dir}/resources"
os.makedirs(resources_dir, exist_ok=True)

for f in files:
    src = os.path.join(output_dir, f)
    dst = os.path.join(resources_dir, f)
    os.system(f"cp {src} {dst}")

os.chdir(repo_dir)
!git add resources/chat_*.txt
!git commit -m "feat: CAMEL-AI SOC 500K pairs - chat_06 to chat_35" || true
!git push origin main || echo "Push failed"